# Data pipeline smoke test (FineWeb2 Vietnamese, small subset)

See `plans/PLAN.md` (Tru cot 1, question 1.3) and `phases/phase-2-small-train.md`.

Mirrors `setup/download_prepare_data.py` + `vislm/tokenizers/bpe_baseline.py`, but
self-contained (Kaggle can't import the local repo package directly). Pulls a small
FineWeb2 Vietnamese subset via streaming (no full 130GB download) and tokenizes it with
Arm A's tokenizer (vinai/PhoGPT-4B) to sanity-check the pipeline before committing to a
full run on the RTX 24GB machine - this is the `compute=kaggle_debug` smoke-test path
`vislm/args.py`'s config overrides are meant for.


In [ ]:
!pip install -q "transformers==4.46.3" datasets


In [ ]:
import json, os, statistics
from datasets import load_dataset
from transformers import AutoTokenizer

TARGET_GB = 0.05  # ~50MB smoke test subset - override this for the full run
OUT_DIR = "/kaggle/working/fineweb2_vi_debug"
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
# Stream + subsample by raw byte budget (same logic as setup/download_prepare_data.py)
ds = load_dataset("HuggingFaceFW/fineweb-2", "vie_Latn", split="train", streaming=True)

target_bytes = int(TARGET_GB * 1e9)
shard_bytes = int(20 * 1e6)  # 20MB shards for a small debug subset
total, shard_idx, shard_written = 0, 0, 0

def open_shard(i):
    return open(os.path.join(OUT_DIR, f"shard_{i:04d}.jsonl"), "w", encoding="utf-8")

f = open_shard(shard_idx)
n_docs = 0
for row in ds:
    line = json.dumps({"text": row["text"]}, ensure_ascii=False)
    n = len(line.encode("utf-8")) + 1
    f.write(line + "\n")
    total += n
    shard_written += n
    n_docs += 1
    if shard_written >= shard_bytes:
        f.close()
        shard_idx += 1
        shard_written = 0
        f = open_shard(shard_idx)
    if total >= target_bytes:
        break
f.close()
print(f"wrote {total/1e6:.1f} MB, {n_docs} docs, {shard_idx + 1} shard(s) to {OUT_DIR}")


In [ ]:
# Tokenize with Arm A's tokenizer (vinai/PhoGPT-4B) and report basic stats
tok = AutoTokenizer.from_pretrained("vinai/PhoGPT-4B")

def shard_paths(d):
    return sorted(
        os.path.join(d, p) for p in os.listdir(d) if p.startswith("shard_")
    )

doc_lens_tokens, doc_lens_bytes = [], []
for path in shard_paths(OUT_DIR):
    with open(path, encoding="utf-8") as fh:
        for line in fh:
            text = json.loads(line)["text"]
            n_bytes = len(text.encode("utf-8"))
            n_tok = len(tok.encode(text, add_special_tokens=False))
            doc_lens_bytes.append(n_bytes)
            doc_lens_tokens.append(n_tok)

total_bytes = sum(doc_lens_bytes)
total_tokens = sum(doc_lens_tokens)
stats = {
    "n_docs": len(doc_lens_bytes),
    "total_bytes": total_bytes,
    "total_tokens": total_tokens,
    "bytes_per_token": total_bytes / total_tokens,
    "mean_doc_tokens": statistics.mean(doc_lens_tokens),
    "median_doc_tokens": statistics.median(doc_lens_tokens),
}
print(stats)


In [ ]:
out_path = "/kaggle/working/metrics_data_pipeline_smoke_test.jsonl"
with open(out_path, "w") as f:
    f.write(json.dumps({"section": "data_pipeline_smoke_test", "results": stats}) + "\n")
print("wrote", out_path)
